# Ridge Regression (L2 Regularization)

Ridge adds an **L2 penalty** (sum of squared weights) to the least-squares cost:

$$J(\beta) = \|\mathbf{X}\beta - y\|^{2} + \alpha\,\|\beta\|_{2}^{2}$$

Closed-form solution (with standardized features):

$$\hat{\beta} = (\mathbf{X}^{\top}\mathbf{X} + \alpha\mathbf{I})^{-1}\,\mathbf{X}^{\top}y$$

L2 **shrinks** all coefficients toward zero but rarely makes them exactly zero.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Synthetic dataset: size & rooms (correlated) + an irrelevant noise feature
np.random.seed(42)
n = 30
size = np.random.uniform(500, 3000, n)
rooms = size / 400 + np.random.randn(n) * 1.5
noise_feat = np.random.randn(n)
X = np.column_stack([size, rooms, noise_feat])
y = 100 * size + 50 * rooms + np.random.randn(n) * 2000
feature_names = ["size", "rooms", "noise_feat"]


## Custom Ridge implementation (closed form)

We implement Ridge ourselves - no `sklearn` model. With the features and target
**centered**, the optimal weights come from the Normal Equation with the L2 penalty
added to the diagonal:

$$\hat{\beta} = (\mathbf{X}_c^{\top}\mathbf{X}_c + \alpha\mathbf{I})^{-1}\,\mathbf{X}_c^{\top}y_c$$

The penalty is applied to the feature weights only (not the intercept). Centering keeps
the coefficients on the original feature scale and matches `sklearn`'s default behavior.


In [ ]:
# --- Custom Ridge Regression (no sklearn) ---
class RidgeRegression:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, float)
        self.x_mean_ = X.mean(axis=0)
        Xc = X - self.x_mean_               # center features
        yc = y - y.mean()                  # center target
        A = Xc.T @ Xc + self.alpha * np.eye(Xc.shape[1])
        self.coef_ = np.linalg.solve(A, Xc.T @ yc)  # closed-form solution
        self.intercept_ = y.mean() - self.x_mean_ @ self.coef_
        return self

    def predict(self, X):
        return np.asarray(X, float) @ self.coef_ + self.intercept_

# Custom OLS = Ridge with alpha = 0
ols = RidgeRegression(alpha=0.0).fit(X, y)
ridge = RidgeRegression(alpha=10.0).fit(X, y)

coef_df = pd.DataFrame({
    "feature": feature_names,
    "OLS": np.round(ols.coef_, 2),
    "Ridge (a=10)": np.round(ridge.coef_, 2),
})
print(coef_df.to_string(index=False))
print(f"\nIntercept  OLS   : {ols.intercept_:.2f}")
print(f"Intercept Ridge : {ridge.intercept_:.2f}")


## Compare coefficient magnitudes


In [ ]:
x_pos = np.arange(len(feature_names))
width = 0.35

plt.bar(x_pos - width/2, ols.coef_, width, label="OLS", color="gray")
plt.bar(x_pos + width/2, ridge.coef_, width, label="Ridge (a=10)", color="teal")
plt.xticks(x_pos, feature_names)
plt.ylabel("Coefficient (standardized)")
plt.title("Ridge shrinks coefficients toward zero")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## Coefficient path vs. alpha (Ridge signature)

As `alpha` grows, every coefficient is shrunk toward zero - the defining L2 behavior.


In [ ]:
alphas = np.logspace(-1, 3, 20)
paths = np.array([RidgeRegression(alpha=a).fit(X, y).coef_ for a in alphas])

plt.figure(figsize=(7, 4.5))
for j, name in enumerate(feature_names):
    plt.plot(alphas, paths[:, j], marker='o', markersize=3, label=name)
plt.xscale('log')
plt.axhline(0, color='black', linewidth=1)
plt.xlabel('alpha (log scale)')
plt.ylabel('Coefficient')
plt.title('Ridge coefficient path')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## Takeaways

- Ridge (**L2**) shrinks all coefficients but keeps them non-zero.
- The irrelevant `noise_feat` weight is pulled toward 0, reducing overfitting.
- Larger `alpha` -> stronger shrinkage. Use `RidgeCV` to pick `alpha` by cross-validation.
